<a href="https://colab.research.google.com/github/Saransh-Basu-01/Advanced-Pytorch/blob/main/Building_a_simple_Rnn_in_pytorch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch
import torch.nn as nn

class SimpleRNNModel(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim, num_rnn_layers=1):
        """
        Initializes the SimpleRNNModel.

        Args:
            input_dim (int): Dimension of input features per timestep.
            hidden_dim (int): Dimension of the RNN hidden state.
            output_dim (int): Dimension of the final output.
            num_rnn_layers (int): Number of stacked RNN layers. Default is 1.
        """
        super().__init__() # Call the __init__ of the parent class (nn.Module)
        self.hidden_dim = hidden_dim
        self.num_rnn_layers = num_rnn_layers

        # Define the RNN layer
        # batch_first=True means input/output tensors shape: (batch, seq, feature)
        self.rnn = nn.RNN(
            input_size=input_dim,
            hidden_size=hidden_dim,
            num_layers=num_rnn_layers,
            batch_first=True, # Make sure input shape is (batch, seq_len, input_size)
            nonlinearity='tanh' # Default activation
        )

        # Define the output layer (fully connected)
        # It takes the final hidden state of the RNN as input
        self.fc = nn.Linear(hidden_dim, output_dim)

    def forward(self, x):
        """
        Defines the forward pass of the model.

        Args:
            x (torch.Tensor): Input tensor of shape (batch_size, seq_len, input_dim).

        Returns:
            torch.Tensor: Output tensor of shape (batch_size, output_dim).
        """
        # Initialize hidden state with zeros
        # Shape: (num_layers, batch_size, hidden_size)
        batch_size = x.size(0)
        h0 = torch.zeros(self.num_rnn_layers, batch_size, self.hidden_dim).to(x.device)

        # Pass data through RNN layer
        # rnn_out shape: (batch_size, seq_len, hidden_size)
        # hn shape: (num_layers, batch_size, hidden_size)
        rnn_out, hn = self.rnn(x, h0)

        # We only need the hidden state from the last time step of the last layer
        # hn contains the final hidden states for all layers.
        # hn[-1] accesses the final hidden state of the last layer.
        # Shape of hn[-1]: (batch_size, hidden_size)
        last_layer_hidden_state = hn[-1]

        # Pass the last hidden state through the fully connected layer
        # out shape: (batch_size, output_dim)
        out = self.fc(last_layer_hidden_state)

        return out

# --- Example Usage ---

# Define model parameters
INPUT_DIM = 10   # Input feature dimension (e.g., embedding size)
HIDDEN_DIM = 20  # Hidden state dimension
OUTPUT_DIM = 5   # Output dimension (e.g., number of classes)
NUM_LAYERS = 1   # Number of RNN layers

# Create the model
model = SimpleRNNModel(INPUT_DIM, HIDDEN_DIM, OUTPUT_DIM, NUM_LAYERS)
print("Model Architecture:")
print(model)

# Create some dummy input data
BATCH_SIZE = 4
SEQ_LEN = 15
dummy_input = torch.randn(BATCH_SIZE, SEQ_LEN, INPUT_DIM) # Shape: (batch, seq, feature)

# Perform a forward pass
output = model(dummy_input)

print(f"\nInput shape: {dummy_input.shape}")
print(f"Output shape: {output.shape}")

# Verify output shape matches (BATCH_SIZE, OUTPUT_DIM)
assert output.shape == (BATCH_SIZE, OUTPUT_DIM)



Model Architecture:
SimpleRNNModel(
  (rnn): RNN(10, 20, batch_first=True)
  (fc): Linear(in_features=20, out_features=5, bias=True)
)

Input shape: torch.Size([4, 15, 10])
Output shape: torch.Size([4, 5])


In [2]:
import torch

# Example Parameters
seq_len = 20      # Longest sequence length
batch_size = 32   # Number of sequences in the batch
input_features = 100 # Dimension of embedding for each word

# Create a dummy input tensor (e.g., filled with random numbers)
# Shape: (seq_len, batch_size, input_features)
rnn_input = torch.randn(seq_len, batch_size, input_features)

print(f"Standard RNN Input Shape: {rnn_input.shape}")
# Output: Standard RNN Input Shape: torch.Size([20, 32, 100])


Standard RNN Input Shape: torch.Size([20, 32, 100])


In [4]:
import torch
import torch.nn as nn

# Example Parameters (same as before)
seq_len = 20
batch_size = 32
input_features = 100
hidden_size = 50 # Example hidden size for the RNN

# Create a dummy input tensor with batch dimension first
# Shape: (batch_size, seq_len, input_features)
rnn_input_batch_first = torch.randn(batch_size, seq_len, input_features)

# Initialize RNN layer with batch_first=True
rnn_layer = nn.RNN(input_size=input_features, hidden_size=hidden_size, batch_first=True)

# Pass the input through the layer (output shape will also have batch first)
output, hidden_state = rnn_layer(rnn_input_batch_first)

print(f"Batch-First RNN Input Shape: {rnn_input_batch_first.shape}")
print(f"Batch-First RNN Output Shape: {output.shape}")
# Output: Batch-First RNN Input Shape: torch.Size([32, 20, 100])
# Output: Batch-First RNN Output Shape: torch.Size([32, 20, 50])


Batch-First RNN Input Shape: torch.Size([32, 20, 100])
Batch-First RNN Output Shape: torch.Size([32, 20, 50])
